In [ ]:
import os
import shutil
import yaml
import bagpy
from bagpy import bagreader
import pandas as pd
import numpy as np
from scipy.integrate import solve_ivp
from tkinter import Tk     # from tkinter import Tk for Python 3.x
from tkinter.filedialog import askopenfilename
from MPCModelFuncs import differential_mpc_model
from MPCModelFuncs import euler_step
import matplotlib.pyplot as plt

In [ ]:
path_bag_learning = '/home/david/bags/cm_untrained/2023-06-04-10-46-47.bag'
bag_learning = bagreader(path_bag_learning)
LEARNING_MSG = bag_learning.message_by_topic('/control/learning_mpcc/nn_data')
learning_data = pd.read_csv(LEARNING_MSG)

In [ ]:
vel_min = 3.0
vel_learning = np.sqrt(learning_data['car_velocity.velocity.x']**2 + learning_data['car_velocity.velocity.y']**2).to_numpy()
vel_learning_short = vel_learning[vel_learning>vel_min]

vx_learning = learning_data['car_velocity.velocity.x'][vel_learning>vel_min].to_numpy()
vy_learning = learning_data['car_velocity.velocity.y'][vel_learning>vel_min].to_numpy()
yaw_rate_learning = learning_data['car_velocity.velocity.theta'][vel_learning>vel_min].to_numpy()
accel_x_learning = learning_data['car_acceleration.x'][vel_learning>vel_min].to_numpy()
accel_y_learning = learning_data['car_acceleration.y'][vel_learning>vel_min].to_numpy()
progress_learning = learning_data['progress'][vel_learning>vel_min].to_numpy()
train_flag = learning_data['just_trained'][vel_learning>vel_min].to_numpy()
exit_flag = learning_data['lmpc_exit_flag'][vel_learning>vel_min].to_numpy()
dist2centerline = learning_data['dist2centerline'][vel_learning>vel_min].to_numpy()

In [ ]:
result_vel = []
temp_vel = []
result_progress = []
temp_progress = []
result_ax = []
temp_ax = []
result_ay = []
temp_ay = []
result_exit_flag = []
temp_exit_flag = []
result_dist2centerline = []
temp_dist2centerline = []

for i, value in enumerate(train_flag):
    temp_vel.append(vel_learning_short[i])
    temp_progress.append(progress_learning[i])
    temp_ax.append(accel_x_learning[i])
    temp_ay.append(accel_y_learning[i])
    temp_exit_flag.append(exit_flag[i])
    temp_dist2centerline.append(dist2centerline[i])

    if value:
        result_vel.append(temp_vel)
        result_progress.append(temp_progress)
        result_ax.append(temp_ax)
        result_ay.append(temp_ay)
        result_exit_flag.append(temp_exit_flag)
        result_dist2centerline.append(temp_dist2centerline)
        temp_vel= []
        temp_progress = []
        temp_ax = []
        temp_ay = []
        temp_exit_flag = []
        temp_dist2centerline = []
        

# Add the remaining elements if the last value in bool_list is False
if temp_vel:
    result_vel.append(temp_vel)
    result_progress.append(temp_progress)
    result_ax.append(temp_ax)
    result_ay.append(temp_ay)
    result_exit_flag.append(temp_exit_flag)
    result_dist2centerline.append(temp_dist2centerline)

In [ ]:
for i in range(len(result_vel)):
    plt.scatter(result_progress[i],result_vel[i],s=0.1,label=str(i),linewidths=0)
    
plt.title("Velocity vs. Progress Comparison")
plt.xlabel("Progress [m]")
plt.ylabel("Velocity [m/s]")

lgnd = plt.legend(fontsize=10)
for i in range(len(result_vel)):
    lgnd.legendHandles[i]._sizes = [30]
    
plt.savefig("../saved_images/velocity_progress_profile_comparison.eps", dpi=1200)
plt.show()

In [ ]:
mean_progress = progress_learning.mean()
result_lap = []
temp_lap = []

for i in (range(len(progress_learning)-1)):
    temp_lap.append(progress_learning[i])

    # print(progress_learning[i+1] - progress_learning[i])

    if abs(progress_learning[i+1] - progress_learning[i]) > mean_progress:
        result_lap.append(temp_lap)
        temp_lap = []

# Add the remaining elements if the last value in bool_list is False
# if temp_lap:
#     result_lap.append(temp_lap)
    
laps = len(result_lap)
time = []
for i in range(laps):
    time.append(len(result_lap[i])*0.05)
    
plt.title("Time per Full Lap Comparison")
plt.xlabel("Lap Number")
plt.ylabel("Time [s]")
    
plt.savefig("../saved_images/time_per_lap.pdf")
plt.plot(range(2,laps+1),time[1:],marker='o')

In [ ]:
for i in range(len(result_exit_flag)):
    plt.scatter(result_progress[i],result_exit_flag[i],s=0.1,label=str(i),linewidths=0)
    
plt.title("Exit Flag vs. Progress Comparison")
plt.xlabel("Progress [m]")
plt.ylabel("Exit Flag")

lgnd = plt.legend(fontsize=10)
for i in range(len(result_vel)):
    lgnd.legendHandles[i]._sizes = [30]
    
plt.savefig("../saved_images/exit_flag_comparison.eps", dpi=1200)
plt.show()

for i in range(len(result_exit_flag)):
    num_1s = result_exit_flag[i].count(1)
    num_2s = result_exit_flag[i].count(2)
    num_other = len(result_exit_flag[i]) - num_1s - num_2s
    percentage = num_1s/len(result_exit_flag[i])
    print(f"Training cycle: {i}     Flag 1: {num_1s}     Flag 2: {num_2s}     Others: {num_other}     Percentage: {percentage*100:.1f}")

In [ ]:
for i in range(len(result_dist2centerline)):
    plt.scatter(result_progress[i],result_dist2centerline[i],s=0.1,label=str(i),linewidths=0)
    
plt.title("Distance to Centerline vs. Progress Comparison")
plt.xlabel("Progress [m]")
plt.ylabel("Distance to Centerline [m]")

lgnd = plt.legend(fontsize=10)
for i in range(len(result_vel)):
    lgnd.legendHandles[i]._sizes = [30]
    
plt.savefig("../saved_images/dist2centerline_comparison.eps", dpi=1200)
plt.show()